# SwarmMind — CV victim detector (Kaggle GPU)

Trains the 2,789-parameter conv net defined by `swarmmind/perception/cnn.py` on rendered
egocentric camera frames, and exports it as plain numpy arrays.

## What this is for

Perception in this project is real: a robot sees through a detector running on its own
occluded 48×48 camera frame, and nothing above `perception/` may read the simulator's
victim list (CLAUDE.md invariant #3). The detector that ships **today** is
`perception/classical.py` — a colour-band + block-pooling heuristic, pure numpy, no ML
runtime, deterministic.

The classical detector has a known, deliberate weakness. Warm rubble is rendered at
`R − B = 128`, *inside* the victim colour band on purpose (M-7), so colour alone cannot
separate rubble from a casualty and the heuristic falls back on **extent** — which is why
it misses a casualty lying in a rubble field and reports a small isolated warm patch as a
person. Its measured precision collapses with range: 0.96 at 1.5 m, 0.48 at 5.0 m, **0.05
at 6.5 m**; buried casualties are 4/4 at 2 m and 0/4 at 5.5 m. Shape discrimination under
noise is the thing a small conv net should be better at. That is the whole bet.

## It only ships if it wins

`training/gate.py` (TECHNICAL §8) compares the two detectors on **mission outcome** —
score, rescued, found — over held-out seeds, and writes the answer into `SHIPPING.md`.
If the classical detector wins, the classical detector ships and these weights go in a
drawer. **A heuristic winning is a result, not a failure** (CLAUDE.md working rules);
perception is real either way. The val F1 printed below is a training signal, not the
decision.

## The architecture is fixed — do not widen it here

    input   48 × 48 × 3  uint8      the same egocentric frame the classical detector sees
    patch   3  → 12, 4×4, stride 4  → 12 × 12 × 12   ReLU
    conv2   12 → 20, 3×3, pad 1     → 12 × 12 × 20   ReLU
    head    20 → 1,  1×1            → 12 × 12        one logit per 4×4-pixel cell

It was chosen with a stopwatch, not by taste: 46 ms per 500-frame pass against the
classical detector's 5.5 ms, which is 23% of the 5 Hz perception budget and takes the
mission from 1.69× real time to ~1.22×. A 3→24 | 24→32 variant costs 92 ms for a task
that is colour and extent discrimination, not object recognition. The full bench table is
in `cnn.py`'s module docstring. Changing the shape here means changing `ARCH` there, and
re-running that bench on the M1.

## Attaching the dataset

1. Build shards locally: `uv run python scripts/export_frames.py` — one `.npz` per mission
   seed, arrays `frames` (N,48,48,3) uint8 and `labels` (N,12,12) uint8. The contract is
   spelled out in `notebooks/README.md`.
2. Upload them as a Kaggle Dataset (e.g. `swarmmind-frames`).
3. **Add Input** → that dataset, then point `DATA_GLOB` below at it.
4. Settings → **Accelerator: GPU T4 ×2** (only one is used), **Internet: off** — nothing
   here downloads anything.
5. Resuming a dead session also needs the previous version's output attached; see the
   checkpoint cell.

## ⚠ The weight layout — the one mistake nothing downstream catches

`perception/cnn.py` runs the forward pass in **numpy, channels-last**, with weights in
**HWIO** order — `(kh, kw, in, out)` — because the patch embed is literally a reshape and
a matmul:

```python
t = (x.reshape(n, h//k, k, w//k, k, c)
      .transpose(0, 1, 3, 2, 4, 5)
      .reshape(n, h//k, w//k, k*k*c))
return t @ w.reshape(k*k*c, -1) + b
```

PyTorch is the opposite on both counts: **NCHW** activations and **OIHW** weights,
`(out, in, kh, kw)`. So every kernel is transposed on export:

```python
w = conv.weight.detach().cpu().numpy().transpose(2, 3, 1, 0)   # OIHW -> HWIO
```

**A transposed kernel trains fine and detects nothing.** This is not a crash, it is a
silent wrong answer, and nothing downstream flags it:

- `w2` exported as `(kw, kh, in, out)` still has shape `(3, 3, 12, 20)` — exactly what
  `ARCH` expects, so the shape assertions pass.
- `CNNVictimDetector.__init__` checks the key list and the shapes, and loads it happily.
- The swarm then finds nobody, which on a dashboard looks identical to a swarm that is
  merely bad at searching. You would debug the auction for a day.

The **final cell** exists solely to catch this: it reimplements `cnn.py`'s forward pass in
numpy, runs it on the exported arrays, and asserts it matches torch to 1e-4. It then
re-runs it with a deliberately kh/kw-swapped kernel and asserts that it *fails*, so the
check cannot be vacuous. **Do not download the weights until that cell passes.**

The same argument covers `mean` / `std`. Inference re-applies them, so a mismatch between
the normalisation used in training and the one used at inference is silent and total.
They are computed here from the training split, carried in the model as buffers so a
resume cannot lose them, and written into the same `.npz` as the weights.

In [ ]:
# ============================== configuration ==============================
# Everything a run might want to change is here. Kaggle-specific paths are marked.

#: Where the frames live. Searched, not hardcoded.
#:
#: Kaggle mounts a dataset at `/kaggle/input/<slug>`, and the slug comes from the dataset
#: *title* — so a hardcoded path breaks the moment the dataset is called anything else,
#: which is exactly how the first run of this notebook failed. Recursive search over
#: `/kaggle/input` finds the shards whatever the dataset is named, and still works
#: locally.
DATA_GLOB = "/kaggle/input/**/*.npz"

#: KAGGLE PATH — resume. `/kaggle/working` is NOT ambiently persistent (PLAN.md R4,
#: TECHNICAL §7.5): it survives only as a *saved notebook version's output*. To continue a
#: session that died, Save Version, then attach that version's output as an input and
#: re-run. With nothing attached this glob matches nothing and training starts from
#: scratch, which is the correct behaviour for a first run.
RESUME_GLOB = "/kaggle/input/*/ckpt.pt"

WORK = "/kaggle/working"
CKPT_PATH = f"{WORK}/ckpt.pt"
EXPORT_PATH = f"{WORK}/detector.npz"

SEED = 42
VAL_FRAC = 0.25         # fraction of *seeds*, not frames — see the split cell
EPOCHS = 20
BATCH = 256
LR = 3e-3               # 2,789 parameters. This is not a ResNet and does not want 1e-4.
HFLIP = True            # mirror augmentation; the validity argument is in the train cell
POS_WEIGHT_CAP = 50.0   # see the loss cell

#: Seeds `training/gate.py` evaluates on. Training may never touch them (TECHNICAL §8),
#: so the loader refuses outright if the dataset contains one.
HELD_OUT_SEEDS = (101, 102, 103, 104, 105, 106, 107, 108, 109, 110)

In [ ]:
import glob
import os
import re
import time
from pathlib import Path

import numpy as np
import torch
from torch import nn

torch.manual_seed(SEED)

DEV = "cuda" if torch.cuda.is_available() else "cpu"

# Input shape never varies (always 48x48x3), so cuDNN autotune pays for itself instead of
# re-benchmarking every call. It makes this *training run* not bit-reproducible, which is
# fine: the determinism invariant protects rehearsed demo timings, and the demo runs the
# exported weights through numpy on the M1. What must be reproducible is the seed-42
# mission, and nothing in this notebook touches it.
torch.backends.cudnn.benchmark = True

print(f"torch {torch.__version__} on {DEV}",
      torch.cuda.get_device_name(0) if DEV == "cuda" else "")

## Data: whole seeds on one side of the line

Each shard is one mission. Frames come off that mission at 5 Hz from up to 512 robots on
one procedurally generated map, so:

- consecutive frames from one robot are nearly identical,
- two robots looking at the same casualty produce near-duplicate frames,
- and every frame in the mission shares the same map, the same rubble layout, the same
  casualty appearances and the same hazard.

A random **frame-level** split therefore puts a near-copy of almost every validation
frame into training. Val F1 then measures memorisation, reads as a superb result, and does
not survive contact with the gate — which runs whole unseen missions. The split below is
by **seed**, and a shard never straddles the line.

In [ ]:
SEED_RE = re.compile(r"seed[_-]?(\d+)")


def shard_seed(path):
    """Group key for a shard: the mission seed from its filename, else the filename.

    The fallback keeps the split honest for oddly-named shards — one file still never
    straddles the train/val boundary, the grouping is just coarser than it should be.
    """
    m = SEED_RE.search(Path(path).name)
    return int(m.group(1)) if m else Path(path).stem


def _order(k):
    # Ints numerically, strings after them. Plain `key=str` would sort seed 11 before
    # seed 9 — deterministic, but it reads as a bug every time someone looks at it.
    return (0, k, "") if isinstance(k, int) else (1, 0, k)


def find_shards():
    """Every frame shard under the input roots, minus any resume checkpoint's own files.

    Recursive, because the dataset's mount directory is named after whatever you called
    the dataset. If nothing matches, the error lists what IS mounted — "no shards found"
    plus an empty directory listing is a different problem from "no shards found" plus a
    dataset full of the wrong files, and the first run should not have to guess which.
    """
    hits = sorted(set(glob.glob(DATA_GLOB, recursive=True))
                  | set(glob.glob("frames/*.npz")) | set(glob.glob("*.npz")))
    return [h for h in hits if Path(h).name != "detector.npz"]


paths = find_shards()
if not paths:
    roots = sorted(glob.glob("/kaggle/input/*"))
    listing = "\n".join(
        f"    {r}/  ->  {', '.join(sorted(x.name for x in Path(r).iterdir())[:8]) or '(empty)'}"
        for r in roots) or "    (nothing is mounted at /kaggle/input)"
    raise AssertionError(
        f"no .npz frame shards found under {DATA_GLOB}.\n"
        f"  Add Input -> your frames dataset (the seedNN.npz files from "
        f"`scripts/export_frames.py`).\n"
        f"  Currently mounted:\n{listing}"
    )
print(f"found {len(paths)} shards: {', '.join(Path(p).name for p in paths)}")

groups = {}
for p in paths:
    groups.setdefault(shard_seed(p), []).append(p)

# The gate's held-out seeds are the only honest measurement this project has of whether
# the CNN beats the heuristic. Training on one of them does not produce a better detector,
# it produces an unfalsifiable claim — so refuse rather than warn.
leaked = sorted(k for k in groups if isinstance(k, int) and k in HELD_OUT_SEEDS)
assert not leaked, (
    f"shards for gate seeds {leaked} are in the dataset. HELD_OUT_SEEDS "
    f"({HELD_OUT_SEEDS[0]}–{HELD_OUT_SEEDS[-1]}) may never be trained on — TECHNICAL §8. "
    f"Re-export without them."
)

keys = sorted(groups, key=_order)
assert len(keys) >= 2, (
    f"only one group ({keys}) — a by-seed split needs at least two missions. Export more "
    f"seeds, or check that the shard filenames carry their seed."
)
n_val = max(1, round(len(keys) * VAL_FRAC))
val_keys, train_keys = keys[-n_val:], keys[:-n_val]
assert train_keys, f"VAL_FRAC={VAL_FRAC} ate the whole training set"

print(f"{len(paths)} shards, {len(keys)} seeds")
print(f"  train seeds: {train_keys}")
print(f"  val   seeds: {val_keys}")


def load(shard_paths):
    """Concatenate shards, asserting the contract from scripts/export_frames.py."""
    fr, lb = [], []
    for p in shard_paths:
        with np.load(p) as z:
            f, lo = z["frames"], z["labels"]
        assert f.dtype == np.uint8, f"{p}: frames are {f.dtype}, expected uint8"
        assert f.shape[1:] == (48, 48, 3), f"{p}: frames {f.shape}, expected (N,48,48,3)"
        assert lo.shape[1:] == (12, 12), f"{p}: labels {lo.shape}, expected (N,12,12)"
        assert len(f) == len(lo), f"{p}: {len(f)} frames vs {len(lo)} labels"
        fr.append(f)
        # Labels are uint8 occupancy, but tolerate a count-per-cell export: anything
        # non-zero is "a casualty is visible in this cell".
        lb.append((lo > 0).astype(np.uint8))
    return np.concatenate(fr), np.concatenate(lb)


Xtr_np, Ytr_np = load([p for k in train_keys for p in groups[k]])
Xva_np, Yva_np = load([p for k in val_keys for p in groups[k]])

# Frames stay uint8 in host RAM (6,912 bytes each) and are cast per batch on the GPU. As
# float32 the same array is 4x larger and 100k frames would be 2.8 GB before training has
# allocated anything.
# Resident on the accelerator, not streamed to it.
#
# The whole dataset is uint8 and small -- 13.6k frames x 48x48x3 is ~94 MB, against 16 GB
# of T4 -- so keeping it on the device makes every batch a pure GPU gather and removes a
# host-to-device copy per step. Streaming it instead (`Xtr[sel].to(DEV)` inside the loop)
# is the default shape of this code and it is the wrong one here: the transfer and the
# CPU-side fancy-index dominate a model this small, which is all of 2,789 parameters.
#
# Falls back to CPU-resident if it will not fit, because a notebook that dies on an OOM
# after twenty minutes of data prep is worse than a slow one.
def _place(*arrays):
    tensors = [torch.from_numpy(a) for a in arrays]
    if DEV != "cuda":
        return tensors
    need = sum(t.numel() * t.element_size() for t in tensors)
    free, _ = torch.cuda.mem_get_info()
    if need > free * 0.5:
        print(f"  dataset {need / 2**20:.0f} MB vs {free / 2**20:.0f} MB free -- "
              f"keeping it on the host and streaming batches")
        return tensors
    return [t.to(DEV) for t in tensors]


Xtr, Ytr, Xva, Yva = _place(Xtr_np, Ytr_np, Xva_np, Yva_np)

print(f"train {len(Xtr):>7d} frames  positive cells {Ytr_np.mean():.4%}"
      f"  {Xtr_np.nbytes / 2**30:.2f} GiB")
print(f"val   {len(Xva):>7d} frames  positive cells {Yva_np.mean():.4%}"
      f"  {Xva_np.nbytes / 2**30:.2f} GiB")
assert len(Xva) > 0, "the val seeds produced no frames"

In [ ]:
# ---- input normalisation, from the TRAINING split only ----------------------
# Computing these over train+val is a (small) leak, but the real reason to take them from
# train alone is that they are *exported* and re-applied at inference: they have to
# describe the data the weights were fitted to. cnn.py reshapes them to (1,1,1,-1) and
# applies them channels-last, so they are per-channel, RGB order, length 3.
#
# Accumulated in chunks because the obvious one-liner materialises an (N*2304, 3) float64
# array — 5.5 GB for 100k frames, which Kaggle survives right up until the next
# allocation.
_s1 = np.zeros(3, np.float64)
_s2 = np.zeros(3, np.float64)
_n = 0
for i in range(0, len(Xtr_np), 1024):
    c = Xtr_np[i:i + 1024].reshape(-1, 3).astype(np.float64)
    _s1 += c.sum(0)
    _s2 += (c * c).sum(0)
    _n += len(c)

MEAN = (_s1 / _n).astype(np.float32)
# Guard against a dead channel: a zero std is a silent divide-by-zero that produces inf
# activations and a loss of nan on step 1.
STD = np.sqrt(np.maximum(_s2 / _n - (_s1 / _n) ** 2, 1e-6)).astype(np.float32)
print(f"mean {MEAN.round(2)}  std {STD.round(2)}   (per channel, RGB, from train only)")

# ---- class balance ---------------------------------------------------------
# BCEWithLogitsLoss with pos_weight, because the balance is brutal. A casualty occupies a
# handful of a frame's 144 cells and most frames contain none at all, so the positive rate
# is order 1%. Plain BCE's easiest minimum is "predict 0 everywhere": ~99% accuracy, zero
# recall, and the gate then compares an empty detector against a working heuristic.
# pos_weight scales the positive term by neg/pos so both classes contribute comparable
# gradient.
_pos = float(Ytr_np.sum())
_cells = float(Ytr_np.size)
POS_RATE = _pos / _cells
RAW_POS_WEIGHT = (_cells - _pos) / max(_pos, 1.0)

# The cap is deliberate, not timidity. At a 0.3% positive rate the raw ratio is ~330,
# which does not balance the classes so much as invert the problem: the net fires nearly
# everywhere, precision collapses, and the report tracker — which promotes on parallax
# from three vantage points ≥2 m apart (TECHNICAL §3.9.5) — gets handed a phantom field
# instead of casualties. Phantoms are not free: each one costs an investigate trip, and
# M-7 already shows what a low-precision detector does to this swarm at range. Raise the
# cap only with a gate number to show for it.
POS_WEIGHT = min(RAW_POS_WEIGHT, POS_WEIGHT_CAP)
print(f"positive rate {POS_RATE:.4%}  raw pos_weight {RAW_POS_WEIGHT:.1f}"
      f"  using {POS_WEIGHT:.1f} (cap {POS_WEIGHT_CAP})")
assert POS_RATE > 0.0, "no positive cells in training labels — check export_frames.py"

## The model

`ARCH` below is copied from `swarmmind/perception/cnn.py`. Kaggle has no checkout of this
repo and no internet, so the copy is a necessity rather than a choice — **if the two ever
disagree, `cnn.py` is right and this is the bug.**

The constructor asserts that every torch weight is the OIHW permutation of the HWIO shape
`cnn.py` expects. That catches a wrong *channel count* at construction, hours before
export. It does **not** catch a kh/kw swap — both kernels are square, so the swapped
shapes are identical. Only the numeric parity cell at the end catches that one.

In [ ]:
#: Copied from swarmmind/perception/cnn.py. HWIO: (kh, kw, in, out).
ARCH = {
    "w1": (4, 4, 3, 12), "b1": (12,),      # patch embed, stride 4
    "w2": (3, 3, 12, 20), "b2": (20,),     # 3x3, pad 1
    "w3": (1, 1, 20, 1), "b3": (1,),       # 1x1 head
}
EXPORT_KEYS = ("w1", "b1", "w2", "b2", "w3", "b3", "mean", "std")


class VictimNet(nn.Module):
    """ARCH, in torch's layout. Read the weight-layout cell before touching this."""

    def __init__(self, mean, std):
        super().__init__()
        # Layer 1's stride equals its kernel, so patches tile rather than overlap. That is
        # not a detail: it is what lets cnn.py express this layer as one reshape plus one
        # matmul, and it is worth 61 ms -> 46 ms on a 500-frame pass.
        self.c1 = nn.Conv2d(3, 12, kernel_size=4, stride=4, padding=0)
        self.c2 = nn.Conv2d(12, 20, kernel_size=3, stride=1, padding=1)
        self.c3 = nn.Conv2d(20, 1, kernel_size=1)
        # Buffers, not module-level constants, so they ride along in the checkpoint. A
        # resume that recomputed them from a different shard set would train weights
        # against one normalisation and export another — silent, and total.
        self.register_buffer("mean", torch.tensor(np.asarray(mean, np.float32)).view(1, 1, 1, 3))
        self.register_buffer("std", torch.tensor(np.asarray(std, np.float32)).view(1, 1, 1, 3))

    def forward(self, frames_u8):
        """(N,48,48,3) uint8 -> (N,12,12) logits."""
        # Normalise channels-last first, exactly as cnn.py does, THEN permute to NCHW.
        # Identical arithmetic to normalising after the permute, but it keeps the training
        # code and the inference code visibly the same operation in the same order.
        x = (frames_u8.float() - self.mean) / self.std
        x = x.permute(0, 3, 1, 2).contiguous()
        x = torch.relu(self.c1(x))
        x = torch.relu(self.c2(x))
        return self.c3(x)[:, 0]


model = VictimNet(MEAN, STD).to(DEV)

for name, conv in (("w1", model.c1), ("w2", model.c2), ("w3", model.c3)):
    kh, kw, cin, cout = ARCH[name]
    assert tuple(conv.weight.shape) == (cout, cin, kh, kw), (
        f"{name}: torch has {tuple(conv.weight.shape)}, ARCH wants OIHW "
        f"{(cout, cin, kh, kw)} to transpose into HWIO {ARCH[name]}"
    )
    assert tuple(conv.bias.shape) == ARCH["b" + name[-1]]

n_par = sum(p.numel() for p in model.parameters())
print(f"{n_par} parameters")
# 2,789 is the number in cnn.py's docstring and in the perception budget. If this trips,
# the architecture drifted from the one that was benchmarked at 46 ms.
assert n_par == 2789, f"{n_par} parameters, cnn.py says 2789 — architecture drift"

with torch.no_grad():
    probe = model(Xva[:2].to(DEV))
assert probe.shape == (2, 12, 12), f"forward gave {tuple(probe.shape)}, expected (N,12,12)"
print(f"forward OK: {tuple(Xva[:2].shape)} -> {tuple(probe.shape)}")

In [ ]:
crit = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(POS_WEIGHT, device=DEV))
opt = torch.optim.Adam(model.parameters(), lr=LR)

start_epoch, best_f1, best_state, history = 0, -1.0, None, []

# ---- resume (PLAN.md R4) ---------------------------------------------------
# Kaggle sessions die: 9 h cap, or preemption. `/kaggle/working` is not ambiently
# persistent — it survives as a saved version's output, which is why resume works by
# attaching that output as an *input*. This is written before the first run, not after
# losing one.
_resume = sorted(glob.glob(RESUME_GLOB))
if _resume:
    # weights_only=False on purpose: this checkpoint is ours and carries optimizer state.
    # torch >= 2.6 flipped the default and would otherwise reject it.
    ck = torch.load(_resume[0], map_location=DEV, weights_only=False)
    model.load_state_dict(ck["model"])
    opt.load_state_dict(ck["opt"])
    start_epoch = ck["epoch"] + 1
    best_f1, best_state, history = ck["best_f1"], ck["best_state"], ck.get("history", [])
    # The normalisation that the weights were trained against came back as buffers. Take
    # the export values from the model, never from the freshly recomputed MEAN/STD above,
    # which describe whatever shard set this session happens to have attached.
    MEAN = model.mean.detach().cpu().numpy().reshape(3)
    STD = model.std.detach().cpu().numpy().reshape(3)
    print(f"resumed {_resume[0]} at epoch {start_epoch}/{EPOCHS}, best val F1 {best_f1:.4f}")
    print(f"  normalisation taken from the checkpoint: mean {MEAN.round(2)} std {STD.round(2)}")
else:
    print(f"no checkpoint matched {RESUME_GLOB} — training from scratch")

## Metrics: precision, recall, F1 — and why accuracy is not one of them

At this class balance accuracy is not a weak metric, it is an actively misleading one. A
model that outputs zero for all 144 cells of every frame scores whatever the negative rate
is — typically 98–99% — with **zero recall**. It has found nobody. The training loop
prints accuracy anyway, next to that all-negative baseline, purely so the number is
visibly worthless rather than absent.

What matters downstream is the trade the report tracker makes: recall decides whether a
casualty is ever seen, precision decides how many investigate trips are wasted on rubble.
F1 is the single number used to pick the epoch to export; the gate is the one that decides
whether any of it ships.

Thresholding is on the **logit at 0**, which is exactly `sigmoid(logit) >= 0.5`, minus a
sigmoid call over every cell.

In [ ]:
def evaluate(net, X, Y, batch=1024):
    net.eval()
    tp = fp = fn = tn = 0
    loss_sum = 0.0
    with torch.no_grad():
        for i in range(0, len(X), batch):
            xb = X[i:i + batch].to(DEV, non_blocking=True)
            yb = Y[i:i + batch].to(DEV, non_blocking=True).float()
            lg = net(xb)
            loss_sum += float(crit(lg, yb)) * len(xb)
            pred = lg > 0.0          # logit > 0 is exactly p >= 0.5
            pos = yb > 0.5
            tp += int((pred & pos).sum())
            fp += int((pred & ~pos).sum())
            fn += int((~pred & pos).sum())
            tn += int((~pred & ~pos).sum())
    prec = tp / (tp + fp) if tp + fp else 0.0
    rec = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * prec * rec / (prec + rec) if prec + rec else 0.0
    return {"loss": loss_sum / len(X), "precision": prec, "recall": rec, "f1": f1,
            "accuracy": (tp + tn) / max(tp + tn + fp + fn, 1), "tp": tp, "fp": fp, "fn": fn}


_null_acc = 1.0 - float(Yva.float().mean())
print(f"val positive rate {1 - _null_acc:.4%} — a net that outputs 0 everywhere scores "
      f"{_null_acc:.4%} accuracy with zero recall.\n"
      f"That is why the accuracy column below is labelled 'ignore'.\n")

g = torch.Generator().manual_seed(SEED)

for ep in range(start_epoch, EPOCHS):
    model.train()
    t0 = time.time()
    run = 0.0
    perm = torch.randperm(len(Xtr), generator=g)
    for i in range(0, len(perm), BATCH):
        sel = perm[i:i + BATCH].to(Xtr.device)
        xb = Xtr[sel].to(DEV, non_blocking=True)
        yb = Ytr[sel].to(DEV, non_blocking=True).float()

        if HFLIP:
            # Mirror augmentation, and it is valid *here* for a specific reason: image
            # column is bearing (perception/camera.py builds one fixed bearing per column,
            # symmetric about the optical axis) and the appearance raster is top-down with
            # no directional lighting, so a left-right mirror is a scene the camera could
            # genuinely have photographed. The label grid flips the same way — 12 cells of
            # 4 px each tile the 48 px width exactly, so the cells stay aligned.
            # A VERTICAL flip is NOT valid: row is range, row 0 is farthest, and near and
            # far are not interchangeable.
            m = (torch.rand(len(sel), generator=g) < 0.5).to(DEV)
            xb[m] = torch.flip(xb[m], dims=[2])      # (N,H,W,C): W is dim 2
            yb[m] = torch.flip(yb[m], dims=[2])      # (N,H,W):   W is dim 2

        opt.zero_grad(set_to_none=True)
        loss = crit(model(xb), yb)
        loss.backward()
        opt.step()
        run += float(loss) * len(sel)

    tr_loss = run / len(perm)
    va = evaluate(model, Xva, Yva)
    history.append({"epoch": ep, "train_loss": tr_loss, **va})

    if va["f1"] > best_f1:
        best_f1 = va["f1"]
        # Detach + clone to CPU: keeping a live reference would track the model as it
        # keeps training, and "best" would silently become "last".
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

    # Checkpoint EVERY epoch, from the first one (PLAN.md R4, TECHNICAL §7.5). best_state
    # rides along so a resume cannot forget which epoch was the good one.
    torch.save({"model": model.state_dict(), "opt": opt.state_dict(), "epoch": ep,
                "best_f1": best_f1, "best_state": best_state, "history": history,
                "arch": ARCH, "pos_weight": POS_WEIGHT,
                "train_seeds": [str(k) for k in train_keys],
                "val_seeds": [str(k) for k in val_keys]},
               CKPT_PATH)

    print(f"epoch {ep + 1:2d}/{EPOCHS}  train {tr_loss:.4f}  val {va['loss']:.4f}  "
          f"P {va['precision']:.3f}  R {va['recall']:.3f}  F1 {va['f1']:.3f}  "
          f"(acc {va['accuracy']:.4f}, ignore)  "
          f"tp {va['tp']:>6d} fp {va['fp']:>7d} fn {va['fn']:>6d}  "
          f"{time.time() - t0:.1f}s")

print(f"\nbest val F1 {best_f1:.4f}")

## Export

Exports the **best-F1 epoch**, not the last one, with the HWIO transpose applied and the
normalisation alongside. `CNNVictimDetector.__init__` checks for exactly these eight keys
— `w1, b1, w2, b2, w3, b3, mean, std` — and raises on a missing one, so the assertion here
fails on Kaggle rather than on the laptop.

In [ ]:
assert best_state is not None, (
    "no epoch completed, so there is no best state to export. If this is a resume, the "
    "checkpoint's epoch is already >= EPOCHS — raise EPOCHS or re-run from scratch."
)
model.load_state_dict(best_state)
model.eval()


def hwio(conv):
    """OIHW (torch) -> HWIO (cnn.py). THE transpose. See the warning cell at the top."""
    w = conv.weight.detach().cpu().numpy().transpose(2, 3, 1, 0)
    return np.ascontiguousarray(w).astype(np.float32)


def bias(conv):
    return conv.bias.detach().cpu().numpy().astype(np.float32)


export = {
    "w1": hwio(model.c1), "b1": bias(model.c1),
    "w2": hwio(model.c2), "b2": bias(model.c2),
    "w3": hwio(model.c3), "b3": bias(model.c3),
    # Straight off the model's buffers, so these are the numbers the weights were trained
    # against even on a resume.
    "mean": model.mean.detach().cpu().numpy().reshape(3).astype(np.float32),
    "std": model.std.detach().cpu().numpy().reshape(3).astype(np.float32),
}

for k, shp in ARCH.items():
    assert export[k].shape == shp, f"{k}: exported {export[k].shape}, cnn.py wants {shp}"
assert export["mean"].shape == (3,) and export["std"].shape == (3,)
assert tuple(sorted(export)) == tuple(sorted(EXPORT_KEYS)), (
    f"exported keys {sorted(export)} != {sorted(EXPORT_KEYS)} — "
    f"CNNVictimDetector.__init__ checks this exact list"
)

np.savez(EXPORT_PATH, **export)
print(f"wrote {EXPORT_PATH}  ({os.path.getsize(EXPORT_PATH) / 1024:.1f} KB)")
for k in EXPORT_KEYS:
    print(f"  {k:>5s}  {str(export[k].shape):>16s}  "
          f"min {export[k].min():+.3f}  max {export[k].max():+.3f}")

## Self-check: the numpy forward pass, here, before the weights leave Kaggle

`_conv` and `_patch_embed` below are copied verbatim from `swarmmind/perception/cnn.py`.
This cell runs them on the exported `.npz` and compares against torch on a real validation
batch.

Three assertions, in order of what they catch:

1. **Parity** — max absolute difference under 1e-4. A layout error is an O(1) difference,
   not a rounding one; the float32 accumulation orders differ between BLAS and cuDNN, so
   expect something around 1e-6.
2. **Negative control** — the same comparison with kh/kw deliberately swapped, asserted to
   **fail**. Both kernels are square, so the swap keeps every shape legal and passes every
   other check in this notebook. If this assertion ever stops firing, assertion 1 has
   become vacuous and is no longer protecting anything.
3. **Sanity** — the fraction of cells the exported detector fires on. A detector that
   fires on 0% or on 90% of cells is broken regardless of how well it matches torch.

**If cell 1 fails, do not download the weights.** Fix the transpose and re-export; the
training run itself is fine.

In [ ]:
# ---- verbatim from swarmmind/perception/cnn.py -----------------------------
# Copies, deliberately: Kaggle has no checkout and no internet. If you edit one, edit
# both — the whole value of this cell is that it is the code that will actually run the
# weights on the laptop.

def _conv(x, w, b, stride, pad):
    """(N,H,W,Cin) x (kh,kw,Cin,Cout) -> (N,H',W',Cout), accumulated over kernel offsets."""
    kh, kw, cin, cout = w.shape
    if pad:
        x = np.pad(x, ((0, 0), (pad, pad), (pad, pad), (0, 0)))
    n, h, w_, _ = x.shape
    oh = (h - kh) // stride + 1
    ow = (w_ - kw) // stride + 1

    out = np.empty((n, oh, ow, cout), dtype=np.float32)
    out[...] = b
    for di in range(kh):
        for dj in range(kw):
            sl = x[:, di:di + oh * stride:stride, dj:dj + ow * stride:stride, :]
            out += sl @ w[di, dj]
    return out


def _patch_embed(x, w, b):
    """First layer, where stride == kernel: patches tile, so this is a reshape."""
    n, h, w_, c = x.shape
    k = w.shape[0]
    t = (x.reshape(n, h // k, k, w_ // k, k, c)
          .transpose(0, 1, 3, 2, 4, 5)
          .reshape(n, h // k, w_ // k, k * k * c))
    return t @ w.reshape(k * k * c, -1) + b


def numpy_logits(z, frames_u8):
    """cnn.py's CNNVictimDetector.logits, on a dict/NpzFile of exported arrays."""
    x = (frames_u8.astype(np.float32) - z["mean"].reshape(1, 1, 1, -1)) / z["std"].reshape(1, 1, 1, -1)
    x = np.maximum(_patch_embed(x, z["w1"], z["b1"]), 0.0)
    x = np.maximum(_conv(x, z["w2"], z["b2"], stride=1, pad=1), 0.0)
    return _conv(x, z["w3"], z["b3"], stride=1, pad=0)[..., 0]


# ---- 1. parity -------------------------------------------------------------
z = dict(np.load(EXPORT_PATH))
n_check = int(min(256, len(Xva)))
frames = Xva[:n_check].numpy()

with torch.no_grad():
    ref = model(Xva[:n_check].to(DEV)).cpu().numpy()
got = numpy_logits(z, frames)

assert got.shape == ref.shape == (n_check, 12, 12), f"{got.shape} vs {ref.shape}"
d = float(np.abs(got - ref).max())
print(f"max |numpy - torch| = {d:.3e} over {n_check} frames, "
      f"logits in [{ref.min():.2f}, {ref.max():.2f}]")
assert d < 1e-4, (
    f"numpy and torch disagree by {d:.3e}. This is almost certainly the weight layout: "
    f"cnn.py wants HWIO (kh,kw,in,out), torch stores OIHW (out,in,kh,kw), and the export "
    f"must transpose(2,3,1,0). DO NOT download these weights."
)

# ---- 2. negative control ---------------------------------------------------
# Swap kh/kw on the two square kernels. Every shape stays legal, ARCH still matches, and
# CNNVictimDetector would load it without complaint — it is simply a different detector.
bad = dict(z)
bad["w1"] = np.ascontiguousarray(
    model.c1.weight.detach().cpu().numpy().transpose(3, 2, 1, 0)).astype(np.float32)
bad["w2"] = np.ascontiguousarray(
    model.c2.weight.detach().cpu().numpy().transpose(3, 2, 1, 0)).astype(np.float32)
assert bad["w1"].shape == ARCH["w1"] and bad["w2"].shape == ARCH["w2"], (
    "the swapped kernels should still satisfy ARCH — that is the point"
)
d_bad = float(np.abs(numpy_logits(bad, frames) - ref).max())
print(f"kh/kw-swapped control: max |numpy - torch| = {d_bad:.3e}  (must be large)")
assert d_bad > 1e-3, (
    "a kh/kw-swapped kernel produced the same output as the correct one, so the parity "
    "check above proves nothing. Most likely the batch is degenerate (all-black frames) "
    "or the kernels are symmetric. Investigate before trusting the export."
)

# ---- 3. sanity -------------------------------------------------------------
fire = float(np.mean(got > 0.0))
label_rate = float(Yva[:n_check].float().mean())
print(f"exported detector fires on {fire:.3%} of cells; labels are {label_rate:.3%} positive")
assert 0.0 < fire < 0.5, (
    f"the exported detector fires on {fire:.2%} of cells. Below 0 it finds nobody; above "
    f"50% it is a phantom generator. Neither is worth taking to the gate."
)

print("\nself-check passed — detector.npz is safe to download.")

## After this notebook

1. **Save Version.** This is also what makes `/kaggle/working` persist — it is the only
   thing that does (R4). Attach *this* version's output as an input to resume.
2. Download `detector.npz` from the version's Output and put it at
   `assets/models/detector.npz` — that is `cnn.py`'s `DEFAULT_WEIGHTS`.
3. Confirm it loads and runs on the laptop, **without torch**:

   ```bash
   uv run python -c "
   import numpy as np
   from swarmmind.perception.cnn import CNNVictimDetector
   d = CNNVictimDetector()
   print(d.logits(np.zeros((4, 48, 48, 3), np.uint8)).shape)   # (4, 12, 12)
   "
   uv run python -m pytest tests/test_perception.py -q
   ```

4. **Run the gate.** It, not the val F1 above, decides what ships:

   ```bash
   uv run python -m swarmmind.training.gate --seeds 10 --report SHIPPING.md
   ```

   The comparison is on mission outcome — score, rescued, found — over
   `HELD_OUT_SEEDS = (101…110)`, not on cell F1. The two detectors do not even share an
   output grid (12×12 here, 8×8 block pooling in the classical one); grid agreement was
   never the question.

5. Watch the **cost**, not just the score. The CNN is 46 ms per 500-frame pass against the
   classical 5.5 ms, which takes the mission from 1.69× real time to ~1.22×. Re-measure
   on the M1 and record it.

6. Add a new `M-` entry to `docs/MEASUREMENTS.md` with the gate numbers and the latency.
   **Never delete a row** — the trend across days is what shows whether a change regressed.

7. If the classical detector wins, **it ships**, `SHIPPING.md` says so, and that is a
   result worth stating plainly. Perception is real either way.